In [ ]:
import os
import sys
from pathlib import Path

# En Serverless, indicar la raíz absoluta del proyecto completo en /Workspace/...
# Puede fijarse aquí o mediante RISKVISION_PROJECT_ROOT; no debe apuntar a notebooks/.
PROJECT_ROOT_OVERRIDE = os.environ.get("RISKVISION_PROJECT_ROOT")
if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser()
    if not project_root.is_absolute():
        raise ValueError("RISKVISION_PROJECT_ROOT debe ser una ruta absoluta")
    project_root = project_root.resolve()
else:
    # Compatibilidad local: iniciar desde la raíz o desde notebooks/.
    candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
    project_root = next((p for p in candidates if (p / "src" / "proyecto").is_dir()), None)
if project_root is None or not (project_root / "src" / "proyecto" / "config.py").is_file():
    raise RuntimeError("No se encuentra src/proyecto: configura RISKVISION_PROJECT_ROOT con la raíz absoluta")
source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import proyecto
if Path(proyecto.__file__).resolve().parent != project_root / "src" / "proyecto":
    raise RuntimeError("El kernel tiene cargada otra copia de proyecto; reinícialo y revisa la raíz")
print("Raíz verificada:", project_root)
print("Módulos:", Path(proyecto.__file__).resolve().parent)
print("Python:", sys.version.split()[0], "| plataforma:", sys.platform)

# El entorno se configura desde Environment ANTES de ejecutar esta celda.
# No se instalan paquetes ni se modifican métodos internos de pandas.
from proyecto.runtime import diagnose_environment
workspace_environment = ("DATABRICKS_RUNTIME_VERSION" in os.environ
                         or project_root.as_posix().startswith(("/Workspace/", "/Volumes/")))
ENVIRONMENT_MODE = os.environ.get("RISKVISION_ENVIRONMENT_MODE",
                                  "serverless" if workspace_environment else "local")
if ENVIRONMENT_MODE not in {"local", "serverless"}:
    raise ValueError("RISKVISION_ENVIRONMENT_MODE debe ser local o serverless")
if workspace_environment and ENVIRONMENT_MODE != "serverless":
    raise ValueError("En Databricks se exige el diagnóstico serverless")
ENVIRONMENT_DIAGNOSTIC = diagnose_environment(
    project_root / "requirements-serverless-06.txt", strict=ENVIRONMENT_MODE == "serverless")
print("Diagnóstico previo a datos:", ENVIRONMENT_DIAGNOSTIC)

from proyecto.models import validation as validation
from proyecto.tracking.reporting import code_revision, log_provisional_report

# Permisos independientes: editar requiere autorización y ejecutar desde esta celda.
ALLOW_REAL_FIT = False
ALLOW_REAL_CALIBRATION = False
ALLOW_TEST_EVALUATION = False
ALLOW_DEFINITIVE_EXPORT = False
CONTROLS = validation.ExecutionControls(
    fit_real=ALLOW_REAL_FIT, calibrate_real=ALLOW_REAL_CALIBRATION,
    evaluate_test=ALLOW_TEST_EVALUATION, export_definitive=ALLOW_DEFINITIVE_EXPORT)
ENABLE_MLFLOW = True
PLOT_CALIBRATION = False
print("Permisos de ejecución:", CONTROLS)


# 07 · Validación e interpretación — RiskVision, Grupo 6

Implementación preparada, sin resultados reales de esta etapa. LightGBM con paquete E y parámetros originales sigue siendo **provisional**. La optimización propia de 12 ensayos aún no se ejecutó. La conclusión de origin/Estefania es un antecedente de otro experimento, no un resultado de nuestra búsqueda.

**La prueba original ya se consultó y el EDA exploró el conjunto completo.** Una evaluación posterior será complementaria sobre datos expuestos, nunca una prueba final completamente independiente. No cambiar modelo, parámetros, calibración ni umbral según esa evaluación. No se crea una marca ficticia de primera apertura.

En una sesión nueva de Serverless: seleccionar la rama revisada, entorno Python 3.12.3 y la dependencia `-r /Workspace/<raíz>/requirements-serverless-07.txt` en Environment, aplicar y reiniciar Python. Hereda las versiones verificadas de la etapa 06; matplotlib 3.11.2 se propone para curvas (verificado localmente, pendiente en Serverless). SHAP no es obligatorio ni se instala automáticamente.

Configurar RISKVISION_PROJECT_ROOT y, si hace falta, RISKVISION_TRAIN_PATH. Cuando Git no esté disponible, indicar RISKVISION_CODE_BRANCH y RISKVISION_CODE_COMMIT. El diagnóstico de la primera celda ocurre antes de leer datos. Se conserva el modo local. No pegar tokens ni ejecutar celdas de instalación temporales.

Los cuatro permisos comienzan en False. Las funciones reutilizables también verifican esos permisos: un comentario o una variable de resultados no bastan para autorizar una operación. Esta barrera protege el flujo previsto; no pretende impedir que una persona programe una lectura externa por fuera del notebook.


## 0. Cargar únicamente el entrenamiento

Se comprueba la huella aprobada antes de deserializar, el esquema de 25 columnas, objetivo binario e ID únicos. No se limpia de nuevo ni se modifica el archivo. No se carga la prueba en esta sección.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

from proyecto.config import ID_COL, TARGET, PROCESSED_DIR
from proyecto.features.build import PACKAGE_E

TRAIN_PATH = Path(os.environ.get("RISKVISION_TRAIN_PATH", str(PROCESSED_DIR / "train.joblib"))).expanduser()
if not TRAIN_PATH.is_absolute():
    raise ValueError("La ruta del entrenamiento debe ser absoluta")
EXPECTED_TRAIN_SHA256 = "4e71503c0072bcbf3ec76cf862614c2e322a79a08e21db02f3225b9398028568"
train = validation.load_training(TRAIN_PATH, expected_sha256=EXPECTED_TRAIN_SHA256)
TRAIN_SHA256 = train.attrs["source_sha256"]
CODE_REVISION = code_revision(project_root,
    branch=os.environ.get("RISKVISION_CODE_BRANCH"), commit=os.environ.get("RISKVISION_CODE_COMMIT"))
VERSIONS = {**ENVIRONMENT_DIAGNOSTIC["versions"], "python": ENVIRONMENT_DIAGNOSTIC["python"]}
print("Entrenamiento:", train.shape, "| SHA256:", TRAIN_SHA256)
print("Procedencia del código:", CODE_REVISION)


## 1. Registro de decisión provisional y protocolo

El registro documenta el modelo y el origen de los parámetros; **no congela una decisión definitiva**. La referencia logística es explicable y usa el mismo paquete E. El LightGBM optimizado se admite únicamente con aprobación y evidencia completa compatible con nuestro protocolo. El JSON de Estefanía selecciona parámetros originales y no se importa como si contuviera nuestro candidato optimizado.

Protocolo propuesto antes de resultados: separar 75/25, estratificado, semilla 7. En el 75 %, obtener una predicción fuera de muestra por cliente con 5 pliegues, semilla 42. No se usan repeticiones como observaciones independientes. Las métricas OOF participan en selección; las de reserva son una comprobación interna condicionada por la exposición anterior. ROC AUC es principal; nunca seleccionar por la prueba complementaria.

**Decisión del equipo:** se adopta la configuración conservada por Estefanía (origin/Estefania, e4067168e4f75dfe3c4f9f5e83cf20df797775c5). Su autoría, métricas reportadas y límites se registran en antecedent y reports/06_adopcion_estefania.md. own_real_optimization_executed permanece False. La validación 07 todavía es propia y pendiente.


In [ ]:
N_SPLITS = 5
SEED = 42
RESERVE_SEED = 7
N_BOOTSTRAP = 1000
APPROVED_OPTIMIZED_REPORT = None
APPROVE_OPTIMIZED = False

DECISION_PROVISIONAL = validation.provisional_decision(
    revision=CODE_REVISION, versions=VERSIONS, training_sha256=TRAIN_SHA256)
DECISION_PROVISIONAL["validation_protocol"] = {
    "reserve_share": .25, "reserve_seed": RESERVE_SEED,
    "oof_splits": N_SPLITS, "seed": SEED, "bootstrap_draws": N_BOOTSTRAP,
    "calibration": "sigmoid con tres pliegues internos y ensemble=False",
    "threshold_grid": sorted(set([*np.linspace(.05, .95, 19), 1 / 6])),
    "approval": "Protocolo propuesto; autorizar ejecución antes de utilizar datos reales",
}
FACTORIES = validation.candidate_factories(
    optimized_evidence=APPROVED_OPTIMIZED_REPORT, approve_optimized=APPROVE_OPTIMIZED,
    training_sha256=TRAIN_SHA256)
print("Candidatos preparados, sin ajuste:", list(FACTORIES))

DECISION_PROVISIONAL["candidate_parameters"] = {
    name: factory().named_steps["model"].get_params() for name, factory in FACTORIES.items()}
if APPROVED_OPTIMIZED_REPORT is not None:
    DECISION_PROVISIONAL["optimized_evidence"] = {
        key: APPROVED_OPTIMIZED_REPORT.get(key) for key in ("report_id", "code", "experiment", "training_sha256")}
    DECISION_PROVISIONAL["own_real_optimization_executed"] = True

display(DECISION_PROVISIONAL)


## 2. Calibración y selección dentro del entrenamiento

La calibración sigmoid se ajusta mediante CV de tres pliegues **dentro del entrenamiento de cada pliegue OOF**. La reserva no ajusta modelos, imputadores, codificadores o calibradores. ensemble=False evita mezclar la comparación con un promedio final de varios modelos.

Se preparan curvas, Brier, log loss y ECE. Criterio propuesto de recomendación: Brier y log loss menores, sin descenso de ROC AUC OOF. Es una recomendación descriptiva pendiente de revisión, no una promoción automática ni una prueba de superioridad.

El análisis de umbral minimiza la penalización hipotética 5:1 en la grilla escrita antes; desempata por menor carga y mayor umbral. Solo admite predicciones OOF del entrenamiento. El umbral 1/6 es una referencia teórica que depende de calibración y costos supuestos; no representa ahorro observado. Las recomendaciones permanecen separadas de los parámetros operativos aprobados.

La celda siguiente prepara conjuntamente OOF, ajustes con el 75 % y comprobación en el 25 %. Nada se ejecuta con ALLOW_REAL_FIT=False. La calibración necesita además ALLOW_REAL_CALIBRATION=True. Todos los modelos permanecen provisionales.


In [ ]:
INTERNAL = None
if not CONTROLS.fit_real:
    print("Ajuste interno bloqueado. Sin métricas reales de etapa 07.")
else:
    if not CODE_REVISION["branch"] or not CODE_REVISION["commit"]:
        raise RuntimeError("Configura la rama y el commit antes de ajustar modelos reales")
    INTERNAL = validation.evaluate_internal(
        train, controls=CONTROLS, optimized_evidence=APPROVED_OPTIMIZED_REPORT,
        approve_optimized=APPROVE_OPTIMIZED, n_splits=N_SPLITS,
        seed=SEED, reserve_seed=RESERVE_SEED, n_boot=N_BOOTSTRAP)
    print("Selección OOF (no estimación independiente):")
    display(INTERNAL["oof_metrics"])
    print("Comprobación en reserva interna expuesta:")
    display(INTERNAL["metrics"])
    display(INTERNAL["recommendations"])
    display(INTERNAL["thresholds"])
    for name, oof in INTERNAL["oof"].items():
        print("Análisis de umbral exclusivamente OOF:", name)
        display(validation.threshold_analysis(oof["y_true"], oof["proba"])[
            ["threshold", "roc_auc", "penalty", "precision", "recall", "flagged_share"]])
    for name, diagnostic in INTERNAL["calibration"].items():
        print("Curva OOF:", name, "| Brier:", diagnostic["brier"], "| log loss:", diagnostic["log_loss"])
        display(diagnostic["curve"])
    if PLOT_CALIBRATION:
        display(validation.plot_calibration(INTERNAL["calibration"]))
    if not CONTROLS.calibrate_real:
        print("Calibración real bloqueada: la comparación antes/después sigue pendiente.")


## 3. Congelar recomendaciones y preparar traspaso; sin ajuste final

07 solo valida internamente, compara, calibra y recomienda umbrales con OOF del desarrollo. Después de revisión explícita se congela FROZEN_SETTINGS. Esto no aprueba un modelo definitivo. **08 es el único responsable del ajuste final con los 24.000 clientes.**

El contrato JSON incorpora decisión, configuración congelada, huella del entrenamiento, versiones, revisión, esquema y copia/huella del informe 07. Autorizar su creación es un permiso para el ajuste final de 08, separado de la aprobación definitiva. No se escribe ni autoriza por defecto.


In [ ]:
APPROVE_CONFIGURATION_FOR_EVALUATION = False
FROZEN_SETTINGS = None
FROZEN_DECISION = None
FITTED_FOR_EVALUATION = {}  # 07 ya no ajusta con todo el entrenamiento
AUTHORIZE_FINAL_FIT_HANDOFF = False
HANDOFF_APPROVED_BY = None
HANDOFF_APPROVED_AT_UTC = None
HANDOFF_OUTPUT = None  # JSON NUEVO; completar solo tras revisión y autorización
HANDOFF_DECISION = HANDOFF_RECEIPT = None
if APPROVE_CONFIGURATION_FOR_EVALUATION:
    FROZEN_DECISION = validation.freeze_for_evaluation(
        DECISION_PROVISIONAL, FROZEN_SETTINGS, approved=True, candidates=tuple(FACTORIES))
    print("Configuración congelada, todavía provisional; ajuste final exclusivo de 08")
else:
    print("Recomendaciones pendientes de revisión; no se ajustó con 24.000 clientes")


## 4. Evaluación complementaria no incluida en esta ejecución

La prueba original ya fue consultada y no es completamente independiente. No se utiliza para seleccionar parámetros, calibración o umbral. Este flujo 07→08→09 no la carga ni realiza evaluación final. Cualquier evaluación complementaria posterior requiere un protocolo y autorización separados; no se reintroduce un ajuste final en 07.


In [ ]:
COMPLEMENTARY = None
if CONTROLS.evaluate_test:
    raise PermissionError("Faltan decisiones y autorización de un protocolo complementario separado; 07 no carga la prueba en este flujo")
print("Evaluación complementaria desactivada; no se leyó el conjunto de prueba")


In [ ]:
if COMPLEMENTARY is None:
    print("Métricas complementarias pendientes; no se inventan resultados.")
else:
    display(COMPLEMENTARY["metrics"])


## 5. Intervalos y comparación pareada

Se compara cada variante contra LightGBM original sobre los mismos clientes. Para ROC AUC se remuestrean ID únicos, se recalculan ambos AUC y se usa variante − original; positivo favorece a la variante. Se omiten remuestras con una sola clase y se informa su número.

La penalización 5:1 tiene **otro bootstrap**, también pareado, que recalcula la penalización media: variante − original negativo favorece a la variante. No confundir signos ni métricas. Son intervalos percentiles del 95 % condicionados a modelos ajustados y clientes observados; no eliminan selección previa, incertidumbre de reentrenar ni exposición histórica. No equivalen a una evaluación independiente.


In [ ]:
if INTERNAL is not None:
    print("Diferencias en la reserva interna (AUC y penalización separadas):")
    display(INTERNAL["comparisons"])
if COMPLEMENTARY is not None:
    print("Diferencias complementarias, con la limitación de exposición previa:")
    display(COMPLEMENTARY["comparisons"])
if INTERNAL is None and COMPLEMENTARY is None:
    print("Intervalos reales pendientes. Implementación comprobada con datos sintéticos.")


## 6. Segmentos y carga de revisión

Análisis descriptivo por SEX y tramos de AGE fijados de antemano: [18,30), [30,45), [45,60), [60,101). Se informa tamaño, frecuencia real, recall, precisión, proporción marcada y discriminación cuando haya ambas clases. Un segmento de una sola clase informa AUC/AP no disponibles, sin inventar una discriminación perfecta.

Se revisa recall al priorizar el 20 % (capacidad supuesta) y carga de revisión. Son asociaciones y limitaciones de un banco y período históricos; no autorizan decisiones discriminatorias ni prueban impacto económico. No se reentrena por los errores de la prueba.


In [ ]:
SEGMENTS = None
if INTERNAL is not None:
    SEGMENTS = validation.segment_metrics(
        INTERNAL["reserve"], INTERNAL["probabilities"][validation.ORIGINAL],
        threshold=INTERNAL["thresholds"][validation.ORIGINAL]["threshold"])
    display(SEGMENTS)
else:
    print("Segmentos y carga pendientes de evaluación autorizada.")


## 7. Interpretación: importancia por permutación

Se mide la disminución de ROC AUC al permutar cada entrada original, con cinco repeticiones, semilla 42 y un hilo. El pipeline completo recalcula las variables derivadas: la importancia de una entrada incorpora su efecto sobre esas derivadas; no es una importancia individual de cada variable del paquete E.

Se utiliza la reserva interna para descripción, no para una nueva selección de variables. No se ajusta el modelo durante la permutación. Variables correlacionadas pueden repartirse la importancia; resultados negativos son posibles. Explica al modelo y no causas. SHAP queda opcional y fuera del camino obligatorio hasta verificar compatibilidad y utilidad.


In [ ]:
IMPORTANCE = None
if INTERNAL is not None:
    IMPORTANCE = validation.interpretation(
        INTERNAL["models"][validation.ORIGINAL], INTERNAL["reserve"], controls=CONTROLS,
        n_repeats=5, seed=SEED)
    display(IMPORTANCE)
else:
    print("Interpretación pendiente; no se ajustó ningún modelo real.")


## 8. Informe provisional y salida para etapas 08 y 09

El informe registra código, versiones, huella de entrenamiento, procedencia, parámetros, permisos y resultados disponibles. Se conserva pendiente de revisión. Con una evaluación autorizada y MLflow habilitado se guarda en un run nuevo, etapa 07, sin sobrescribir informes anteriores. reports/07_validacion.md describe el protocolo y su estado; no contiene cifras inventadas.

La exportación definitiva exige permiso independiente y status=approved_definitive además de configuración congelada. El registro actual es provisional y no cumple esa aprobación: activar un indicador por accidente no lo convierte en definitivo. La escritura usa creación exclusiva, sin sobrescribir modelos existentes. La integración final con 08/09 se realizará después de revisar métricas, configuración y formato del artefacto.


In [ ]:
REPORT_07 = validation.validation_report(
    FROZEN_DECISION or DECISION_PROVISIONAL, controls=CONTROLS,
    internal=INTERNAL, complementary=COMPLEMENTARY, importance=IMPORTANCE, segments=SEGMENTS)
display(REPORT_07)
REPORT_RUN_ID = None
if ENABLE_MLFLOW and (INTERNAL is not None or COMPLEMENTARY is not None):
    from datetime import datetime, timezone
    from uuid import uuid4
    from proyecto.tracking.mlflow_utils import configure_mlflow, resolve_experiment_name, set_experiment
    import mlflow
    if mlflow.active_run() is not None:
        raise RuntimeError("Cierra el run activo antes de registrar el informe 07")
    in_workspace = workspace_environment
    tracking_mode = "databricks" if in_workspace else "local"
    print("Seguimiento:", configure_mlflow(backend=tracking_mode))
    requested = resolve_experiment_name("riskvision-07-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex)
    if mlflow.get_experiment_by_name(requested) is not None:
        raise RuntimeError("El experimento ya existe; no se reutiliza")
    full_name = set_experiment(requested)
    experiment = mlflow.get_experiment_by_name(full_name)
    if experiment is None:
        raise RuntimeError("No se recuperó el experimento 07 por su ruta completa")
    REPORT_RUN_ID = log_provisional_report(REPORT_07, experiment.experiment_id, stage="07")
    print("Informe provisional:", full_name, "| run:", REPORT_RUN_ID)
if CONTROLS.export_definitive:
    raise PermissionError("La persistencia y el único ajuste final pertenecen a 08")
if AUTHORIZE_FINAL_FIT_HANDOFF:
    if FROZEN_DECISION is None or INTERNAL is None or HANDOFF_OUTPUT is None:
        raise PermissionError("Se requiere validación 07, configuración revisada y destino JSON nuevo")
    from proyecto.models.decision import prepare_handoff, write_handoff
    HANDOFF_DECISION = prepare_handoff(FROZEN_DECISION, REPORT_07,
        authorize_final_fit=True, approved_by=HANDOFF_APPROVED_BY, approved_at=HANDOFF_APPROVED_AT_UTC)
    HANDOFF_RECEIPT = write_handoff(HANDOFF_DECISION, HANDOFF_OUTPUT)
    print("Contrato provisional para 08:", HANDOFF_RECEIPT)
else:
    print("Traspaso sin autorizar; no se creó archivo ni se aprobó modelo definitivo")
